# Loss Functions

In this exercise, you will compare the effects of Loss functions on a `LinearRegression` model.

👇 Let's download a CSV file to use for this challenge and parse it into a DataFrame

In [1]:
import pandas as pd

data = pd.read_csv("https://wagon-public-datasets.s3.amazonaws.com/05-Machine-Learning/04-Under-the-Hood/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
223,0.71,710.5,269.5,220.5,3.5,0.10,12.460
346,0.86,588.0,294.0,147.0,7.0,0.25,29.585
30,0.71,710.5,269.5,220.5,3.5,0.00,8.830
623,0.62,808.5,367.5,220.5,3.5,0.40,17.070
72,0.74,686.0,245.0,220.5,3.5,0.10,11.895


🎯 Your task is to predict the average temperature inside a greenhouse based on its design. Your temperature predictions will help you select the appropriate greenhouse design for each plant, based on their climate needs. 

🌿 You know that plants can handle small temperature variations, but are exponentially more sensitive as the temperature variations increase. 

## 1. Theory 

❓ Theoretically, which Loss function would you train your model on to limit the risk of killing plants?

<details>
<summary> 🆘 Answer </summary>
    
By theory, you would use a Mean Square Error (MSE) Loss function. It would penalize outlier predictions and prevent your model from committing large errors. This would ensure smaller temperature variations and a lower risk for plants.

</details>

## 2. Application

### 2.1 Preprocessing

❓ Standardise the features

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select only the features
X = data.loc[:,'Relative Compactness':'Glazing Area']

# Fit scaler
scaler = StandardScaler().fit(X)

# Scale continuous features
X_scaled = scaler.transform(X)

### 2.2 Modeling

In this section, you are going to verify the theory by evaluating models optimized on different Loss functions.

### Least Squares (MSE) Loss

❓ **10-Fold Cross-validate** a Linear Regression model optimized by **Stochastic Gradient Descent** (SGD) on a **Least Squares Loss** (MSE)



In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.linear_model import SGDRegressor

# Squared loss SGD Regressor
sgd_model = SGDRegressor(loss="squared_error")

# Cross Validate Model
sgd_model_cv = cross_validate(
    sgd_model,
    X_scaled,
    data['Average Temperature'],
    cv = 10,
    scoring = ['r2','max_error']
)

sgd_model_cv

{'fit_time': array([0.00361967, 0.00214028, 0.0020082 , 0.00198793, 0.00190282,
        0.00217319, 0.0020628 , 0.00217605, 0.00194216, 0.00211477]),
 'score_time': array([0.00076127, 0.00052571, 0.00047374, 0.00045609, 0.00045514,
        0.00044584, 0.00043893, 0.00043893, 0.00043488, 0.00044322]),
 'test_r2': array([0.78636757, 0.90819692, 0.8952713 , 0.88377037, 0.93106163,
        0.89653904, 0.92715325, 0.91586555, 0.89543045, 0.93957495]),
 'test_max_error': array([-9.8894405 , -8.60940566, -8.83012729, -9.21325243, -8.97055529,
        -8.6094948 , -8.57399688, -8.77219466, -8.43019394, -7.66703189])}

❓ Compute 
- the mean cross-validated R2 score and save it in the variable `r2`
- the single biggest prediction error in °C of all your folds and save it in the variable `max_error_celsius`?

(Tips: `max_error` is an accepted scoring metric in sklearn)

In [4]:
r2 = sgd_model_cv['test_r2'].mean()
r2

0.8979231028531283

In [5]:
max_error_celsius = abs(sgd_model_cv['test_max_error']).max()
max_error_celsius

9.889440497047062

### Mean Absolute Error (MAE) Loss

What if we optimize our model on the MAE instead?

❓ **10-Fold Cross-validate** a Linear Regression model optimized by **Stochastic Gradient Descent** (SGD) on a **MAE** Loss

<details>
<summary>💡 Hints</summary>

- MAE loss cannot be directly specified in `SGDRegressor`. It must be engineered by adjusting the right parameters

</details>

In [ ]:
# MAE loss engineered by setting epsilon_insensitive = 0
mae_model = SGDRegressor(loss="epsilon_insensitive", epsilon = 0)

# Cross Validate Model
mae_sgd = cross_validate(
    mae_model,
    X_scaled,
    data['Average Temperature'],
    cv = 10,
    scoring = ['r2','max_error']
)

❓ Compute 
- the mean cross-validated R2 score, store it in `r2_mae`
- the single biggest prediction error of all your folds, store it in `max_error_mae`?

In [7]:
r2_mae = mae_sgd['test_r2'].mean()
r2_mae

0.875957852349126

In [8]:
max_error_mae = abs(mae_sgd['test_max_error']).max()
max_error_mae

11.241100223949708

## 3. Conclusion

❓Which of the models you evaluated seems the most appropriate for your task?

<details>
<summary> 🆘Answer </summary>
    
Although mean cross-validated r2 scores are approximately similar between the two models, the one optimized on a MAE has more chance to make larger mistakes from time to time, increasing the risk of killing plants!

    
</details>

# 🏁 Check your code and push your notebook

In [9]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform darwin -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /Users/davywai/.pyenv/versions/3.12.9/envs/lewagon/bin/python
cachedir: .pytest_cache
rootdir: /Users/davywai/code/lewagon/wgn-ds-2025/Module05_MachineLearning/W04D01__04-UnderTheHood/solution_05-ML_04-Under-the-hood_01-Loss-Functions/tests
plugins: anyio-4.8.0, typeguard-4.4.2
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.10s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

